In [1]:
from langchain_openai import ChatOpenAI
import os
#from langchain_google_genai import ChatGoogleGenerativeAI

base_url= os.getenv("AICREDITS_BASE_URL")
api_key = os.getenv("AICREDITS_API_KEY")

In [2]:
llm = ChatOpenAI(
    model="google/gemini-2.5-flash",
    api_key=api_key,
    base_url=base_url,
    #temperature=2.0
)

# Test the model
response = llm.invoke("Hello!")
print(response.content)


Hello there! How can I help you today?


# Structured Output

In [6]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate


class Step(BaseModel):
    """A step that is a part of the plan to solve the task."""
    step: str = Field(description="Description of the step")


class Plan(BaseModel):
    """A plan to solve the task."""
    steps: list[Step]


prompt = PromptTemplate.from_template(
    "Prepare a step-by-step plan to solve the given task.\n"
    "TASK:\n{task}\n"
)

In [7]:
result = (prompt | llm.with_structured_output(Plan)).invoke("How to write a bestseller on Amazon about generative AI?")
assert isinstance(result, Plan)
print(f"Amount of steps: {len(result.steps)}")
for step in result.steps:
  print(step.step)
  break

Amount of steps: 18
Identify your target audience: Are you writing for beginners, developers, business leaders, or a general audience? This will dictate your tone, depth, and examples.


In [10]:
##Using JSON MODE



plan_schema = {
    "type": "ARRAY",
    "items": {
        "type": "OBJECT",
          "properties": {
              "step": {"type": "STRING"},
          },
      },
}

query = "How to write a bestseller on Amazon about generative AI?"
result = (prompt | llm.with_structured_output(schema=plan_schema, method="json_mode")).invoke(query)


In [11]:
assert(isinstance(result, list))
print(f"Amount of steps: {len(result)}")
print(result[0])

Amount of steps: 9
{'step': 1, 'title': 'Market Research & Niche Identification', 'description': "Analyze existing bestsellers on Amazon about Generative AI. Identify gaps, unmet needs, or unique angles (e.g., 'Generative AI for non-technical professionals,' 'Ethical considerations in AI art,' 'Prompt Engineering for beginners'). Define your target audience clearly (e.g., developers, artists, business owners, students). Conduct keyword research using tools like Amazon's search bar, KDP Rocket, or Publisher Rocket to find popular search terms related to your niche."}


In [12]:
from langchain_core.output_parsers import JsonOutputParser


plan_schema = {
    "name": "plan_schema",
    "description": "Creates a List of steps to achieve a goal",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "steps": {
                "type": "array",
                "description": "List of steps to achieve the goal",
                "items": {
                    "type": "object",
                    "properties": {
                        "step": {
                            "type": "string",
                            "description": "A step to achieve part of the goal",
                        }
                    },
                    "additionalProperties": False,
                    "required": ["step"],
                },
            }
        },
        "additionalProperties": False,
        "required": ["steps"],
    },
}


response_format = {"type": "json_schema", "json_schema": plan_schema}

llm_json = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=api_key,
    base_url=base_url,
    model_kwargs={"response_format": response_format}
    #temperature=2.0
)

result = (prompt | llm_json | JsonOutputParser()).invoke(query)
assert isinstance(Plan.model_validate(result), Plan)
print(f"Amount of steps: {len(result['steps'])}")
print(result["steps"][0])

Amount of steps: 14
{'step': 'Research the market for best-selling books on generative AI to understand current trends and topics.'}


In [13]:
from langchain_core.output_parsers import StrOutputParser

response_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "sentiment_classifier",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "sentiment": {
                    "type": "string",
                    "enum": ["positive", "negative", "neutral"],
                }
            },
            "required": ["sentiment"],
            "additionalProperties": False,
        },
    },
}

prompt = PromptTemplate.from_template(
    "Classify the tone of the following customer's review" "\n{review}\n"
)

review = "I Like this movie!"
llm_enum = ChatOpenAI(
    model="gpt-4o-mini", 
    base_url=base_url,
    api_key=api_key,
    model_kwargs={"response_format": response_schema}
)
result = (prompt | llm_enum | JsonOutputParser()).invoke(review)
print(result["sentiment"])

positive
